Hands-on Task 2: Build a Review Pipeline with Claude 

Objective 

Create a multi-step LLM pipeline with sequential processing. 

Task 
Define a state structure: 
class ReviewState(TypedDict): 
    product: str 
    review: str 
    sentiment: str 
    reply: str 
`` 
Build the following pipeline: 
START → review_node → sentiment_node → reply_node → END 
Node Requirements 
1. review_node 
Generate a short product review using Claude 
2. sentiment_node 
Classify the review into:  
Positive 
Negative 
Neutral 
3. reply_node 
Generate a one-line brand response based on the sentiment 
Test Input 
product = "wireless noise-cancelling headphones" 
Skills Tested 
Chaining multiple LLM steps 
State management across nodes 
Prompt design for generation and classification 
Using Claude in graph-based workflows 

In [5]:
from dotenv import load_dotenv
import os
from typing import TypedDict
from langchain_anthropic import ChatAnthropic
from langgraph.graph import StateGraph, START, END

# ==========================================
# 1. Load Custom Environment Variables
# ==========================================
load_dotenv()

# Extract your custom variables
api_key = os.environ.get("KEY")
base_url = os.environ.get("BASE_URL")
model_name = os.environ.get("MODEL", "global.anthropic.claude-haiku-4-5-20251001-v1:0")

if not api_key:
    raise ValueError("❌ 'KEY' not found in your environment or .env file.")

# ==========================================
# 2. Initialize Custom ChatAnthropic Client
# ==========================================
llm = ChatAnthropic(
    model=model_name,
    anthropic_api_key=api_key,
    anthropic_api_url=base_url,  # Routes requests to your proxy gateway
    temperature=0.7
)

# ==========================================
# 3. Define the ReviewState
# ==========================================
class ReviewState(TypedDict):
    product: str
    review: str
    sentiment: str
    reply: str

# ==========================================
# 4. Define the Nodes
# ==========================================

def review_node(state: ReviewState) -> dict:
    """Generates a short, realistic product review."""
    prompt = f"Write a brief, realistic consumer review for this product: {state['product']}. Keep it to 2-3 sentences."
    response = llm.invoke(prompt)
    return {"review": response.content}

def sentiment_node(state: ReviewState) -> dict:
    """Classifies the generated review into Positive, Negative, or Neutral."""
    prompt = (
        f"Analyze the sentiment of this product review:\n\n'{state['review']}'\n\n"
        "Respond with exactly one word from these options: Positive, Negative, or Neutral. "
        "Do not include any other text or punctuation."
    )
    # Target exact extraction using zero temperature
    classification_llm = llm.with_config(configurable={"temperature": 0.0})
    response = classification_llm.invoke(prompt)
    
    sentiment = response.content.strip().replace(".", "")
    return {"sentiment": sentiment}

def reply_node(state: ReviewState) -> dict:
    """Generates a polite one-line brand response tailored to the sentiment."""
    sentiment = state["sentiment"]
    review = state["review"]
    
    prompt = (
        f"You are a customer service representative. Based on a {sentiment} review, "
        f"write a professional, one-line brand response to this customer review:\n\n'{review}'"
    )
    response = llm.invoke(prompt)
    return {"reply": response.content}

# ==========================================
# 5. Build and Connect the Graph
# ==========================================
workflow = StateGraph(ReviewState)

# Register nodes
workflow.add_node("generate_review", review_node)
workflow.add_node("analyze_sentiment", sentiment_node)
workflow.add_node("brand_reply", reply_node)

# Connect sequential edges
workflow.add_edge(START, "generate_review")
workflow.add_edge("generate_review", "analyze_sentiment")
workflow.add_edge("analyze_sentiment", "brand_reply")
workflow.add_edge("brand_reply", END)

# Compile graph
app = workflow.compile()

# ==========================================
# 6. Run and Test the Pipeline
# ==========================================
initial_input = {"product": "wireless noise-cancelling headphones"}
final_state = app.invoke(initial_input)

# Display results
print("-" * 50)
print(f"Product: {final_state['product']}\n")
print(f"Generated Review:\n{final_state['review']}\n")
print(f"Detected Sentiment: {final_state['sentiment']}\n")
print(f"Official Brand Reply:\n{final_state['reply']}")
print("-" * 50)

--------------------------------------------------
Product: wireless noise-cancelling headphones

Generated Review:
# Review: Wireless Noise-Cancelling Headphones

These headphones deliver solid noise cancellation for flights and commutes, though the sound quality is good rather than exceptional. Battery life easily gets me through a full week of daily use, and they're comfortable enough for extended wear. My only complaint is the touch controls can be finicky sometimes, but overall they're a solid mid-range option.

Detected Sentiment: Positive

Official Brand Reply:
Thank you for choosing our headphones—we're thrilled they're enhancing your travel and commute experiences, and we appreciate your feedback on the touch controls, which helps us continue improving your listening journey.
--------------------------------------------------
